# SDXL-Turbo embedding bounds: genai_bench

A dataset-specific copy of the original bounds experiment. Computes empirical min/max at every tensor coordinate and overall, for token and pooled embeddings. See README.md for sources and interpretation.

In [ ]:
import sys
from pathlib import Path

EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "dataset_bounds.py").exists():
    EXPERIMENT_DIR = EXPERIMENT_DIR / "pre-testing" / "clamp_range"
sys.path.insert(0, str(EXPERIMENT_DIR.resolve()))
import pandas as pd
import torch
from dataset_bounds import load_result, run_dataset

## Configuration and timing

MPS is selected when available, otherwise CPU. Encoding uses the pinned float16 SDXL text encoders. Sources are text-only. The pilot selects a sample fitting a 50-minute working budget, with a 55-minute normal stop and a 60-minute runner timeout. Existing results for this dataset are replaced when this cell runs.

In [ ]:
DATASET = "genai_bench"
metadata = run_dataset(DATASET)
print(
    {
        key: metadata[key]
        for key in [
            "status",
            "device",
            "unique_prompts",
            "planned_prompts",
            "encoded_prompts",
            "elapsed_seconds",
            "pilot_mean_seconds",
        ]
    }
)

## Overall min/max and summed coordinate spans

Arithmetic uses float32. The span sum describes an axis-aligned empirical range, not linear rank or coverage of the entire embedding space.

In [ ]:
summary, bounds = load_result(DATASET)
display(pd.DataFrame(summary["embeddings"]).T)
print("Truncated prompts by tokenizer:", summary["truncation_counts"])
print("Overlap with Parti among selected prompts:", summary["parti_overlap_selected"])

## Bounds at every position

The saved CPU tensors contain all coordinates: token shape `(1, 77, 2048)` and pooled shape `(1, 1280)`. The comparison notebook produces full and equal-count comparisons, heatmaps, growth curves and a packaged-Parti drift check.

In [ ]:
for kind, tensors in bounds.items():
    print(kind, "shape:", tuple(tensors["min"].shape))
    print("First five lower bounds:", tensors["min"].flatten()[:5])
    print("First five upper bounds:", tensors["max"].flatten()[:5])
    assert torch.isfinite(tensors["min"]).all()
    assert torch.all(tensors["min"] <= tensors["max"])
print("Artifacts:", EXPERIMENT_DIR / "results" / DATASET)